# 04️⃣ Portfolio Construction

Construct portfolios for Conservative, Balanced, and Aggressive profiles strictly in the out-of-sample period (2016-2021). Select top 10 stocks on each day.

In [1]:
import os, pathlib, pandas as pd, numpy as np
%matplotlib inline


In [2]:
NOTEBOOK_DIR = pathlib.Path(os.path.abspath('') if '__file__' not in locals() else os.path.dirname(__file__))
PROJECT_ROOT = NOTEBOOK_DIR.parent
DATA_PARQUET = PROJECT_ROOT / 'data' / 'parquet'
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(DATA_PARQUET / 'modeled_data.parquet')
df['Date'] = pd.to_datetime(df['Date'])

# Evaluate strictly in the out-of-sample test period
test_df = df[df['Date'] >= '2016-01-01'].copy()
test_df = test_df.sort_values(['Date', 'Symbol'])

# Select top 10 long candidates based on predicted return on each date
active_holdings = test_df.groupby('Date').apply(lambda x: x.nlargest(10, 'Predicted_Return')).reset_index(drop=True)
active_holdings.to_parquet(DATA_PARQUET / 'active_holdings.parquet', index=False)

# Active Equity: average return of selected top 10 stocks on day t+1
daily_active_equity = active_holdings.groupby('Date')['Target_Return'].mean().reset_index()
daily_active_equity.rename(columns={'Target_Return': 'Active_Equity_Return'}, inplace=True)

# Passive Equity (NIFTY50 Benchmark): average return of all stocks on day t+1
daily_passive_equity = test_df.groupby('Date')['Target_Return'].mean().reset_index()
daily_passive_equity.rename(columns={'Target_Return': 'Passive_Equity_Return'}, inplace=True)

# Merge returns
portfolio_returns = pd.merge(daily_active_equity, daily_passive_equity, on='Date')

# Merge average predicted return for allocation scaling
daily_pred = active_holdings.groupby('Date')['Predicted_Return'].mean().reset_index()
daily_pred.rename(columns={'Predicted_Return': 'Avg_Pred_Return'}, inplace=True)
portfolio_returns = pd.merge(portfolio_returns, daily_pred, on='Date')

# Define Allocations
portfolio_returns['Conservative_Equity'] = 0.20
portfolio_returns['Conservative_Debt'] = 0.80

portfolio_returns['Balanced_Equity'] = 0.50
portfolio_returns['Balanced_Debt'] = 0.50

# Aggressive: scales around 80% based on prediction strength
portfolio_returns['Aggressive_Equity'] = np.clip(0.8 + portfolio_returns['Avg_Pred_Return'] * 20, 0.50, 0.95)
portfolio_returns['Aggressive_Debt'] = 1.0 - portfolio_returns['Aggressive_Equity']

# Compute active turnover and transaction costs (0.1% = 10 bps)
dates = sorted(portfolio_returns['Date'].unique())
agg_turnovers = []
bal_turnovers = []
prev_agg_weights = {}
prev_bal_weights = {}

for i, dt in enumerate(dates):
    row = portfolio_returns[portfolio_returns['Date'] == dt].iloc[0]
    selected_symbols = active_holdings[active_holdings['Date'] == dt]['Symbol'].tolist()
    
    # Aggressive
    agg_eq = row['Aggressive_Equity']
    agg_weights = {sym: agg_eq / 10.0 for sym in selected_symbols}
    agg_weights['DEBT'] = row['Aggressive_Debt']
    if i == 0:
        agg_turnover = 1.0
    else:
        all_assets = set(agg_weights.keys()).union(set(prev_agg_weights.keys()))
        agg_turnover = sum(abs(agg_weights.get(a, 0.0) - prev_agg_weights.get(a, 0.0)) for a in all_assets) / 2.0
    agg_turnovers.append(agg_turnover)
    prev_agg_weights = agg_weights
    
    # Balanced
    bal_eq = row['Balanced_Equity']
    bal_weights = {sym: bal_eq / 10.0 for sym in selected_symbols}
    bal_weights['DEBT'] = row['Balanced_Debt']
    if i == 0:
        bal_turnover = 1.0
    else:
        all_assets = set(bal_weights.keys()).union(set(prev_bal_weights.keys()))
        bal_turnover = sum(abs(bal_weights.get(a, 0.0) - prev_bal_weights.get(a, 0.0)) for a in all_assets) / 2.0
    bal_turnovers.append(bal_turnover)
    prev_bal_weights = bal_weights

portfolio_returns['Aggressive_Turnover'] = agg_turnovers
portfolio_returns['Aggressive_Transaction_Cost'] = portfolio_returns['Aggressive_Turnover'] * 0.001

portfolio_returns['Balanced_Turnover'] = bal_turnovers
portfolio_returns['Balanced_Transaction_Cost'] = portfolio_returns['Balanced_Turnover'] * 0.001

portfolio_returns['Conservative_Transaction_Cost'] = 0.0

portfolio_returns.to_parquet(DATA_PARQUET / 'portfolio_construction.parquet', index=False)

summary_df = pd.DataFrame({
    'Profile': ['Conservative', 'Balanced', 'Aggressive'],
    'Equity_Allocation': [
        portfolio_returns['Conservative_Equity'].mean(),
        portfolio_returns['Balanced_Equity'].mean(),
        portfolio_returns['Aggressive_Equity'].mean()
    ],
    'Debt_Allocation': [
        portfolio_returns['Conservative_Debt'].mean(),
        portfolio_returns['Balanced_Debt'].mean(),
        portfolio_returns['Aggressive_Debt'].mean()
    ]
})
summary_df.to_csv(OUTPUT_DIR / 'portfolio_summary.csv', index=False)
print('Portfolio construction complete. Allocations summary:')
print(summary_df)


C:\Users\sajal\AppData\Local\Temp\ipykernel_21992\1694265493.py:15: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  active_holdings = test_df.groupby('Date').apply(lambda x: x.nlargest(10, 'Predicted_Return')).reset_index(drop=True)


Portfolio construction complete. Allocations summary:
        Profile  Equity_Allocation  Debt_Allocation
0  Conservative           0.200000         0.800000
1      Balanced           0.500000         0.500000
2    Aggressive           0.828156         0.171844
